In [1]:
from imports import *
from scipy.stats import gaussian_kde

/home/dewan/miniconda3/envs/opti/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env_vars = dotenv_values(dotenv_path="../.env")
with open(f'.{env_vars["data_folder_path"]}/{env_vars["split_fname"]}', "rb") as f:
    data = pickle.load(f)

train_data = data["train_data"]
test_data = data["test_data"]
full_kvasir_data = train_data + test_data

### Plotting the KDE 

In [3]:
def get_mask_area_ratio(data):
    msk_transform = transforms.Compose([
        transforms.Resize(ast.literal_eval(env_vars.get("mask_size")), transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
        ])

    data_with_area = []
    for idx, value in enumerate(data):
        mask = msk_transform(value[1])
        area_ratio = (mask == 1).sum().item() / mask.numel()
        data_with_area.append((value, area_ratio))

    sorted_data_with_area = sorted(data_with_area, key=lambda x: x[1])
    area_ratios = [item[1] for item in sorted_data_with_area]
    return area_ratios

In [4]:
def plot_train_vs_test_kde(train_data, test_data, savefig_name=None, bw=0.5):
    full_dataset = train_data + test_data
    train_ratios = get_mask_area_ratio(train_data)
    test_ratios  = get_mask_area_ratio(test_data)
    full_ratios  = get_mask_area_ratio(full_dataset)

    # KDE
    x_vals = np.linspace(0, 1, 1000)
    train_kde = gaussian_kde(train_ratios, bw_method=bw)
    test_kde  = gaussian_kde(test_ratios,  bw_method=bw)
    full_kde  = gaussian_kde(full_ratios,  bw_method=bw)
    train_y = train_kde(x_vals)
    test_y  = test_kde(x_vals)
    full_y  = full_kde(x_vals)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    ax.plot(x_vals, train_y, color="#E74C3C", linewidth=2, label="Train KDE")
    ax.plot(x_vals, test_y,  color="#27AE60", linewidth=2, label="Test KDE")
    ax.plot(x_vals, full_y,  color="black",   linewidth=2, label="Full Dataset KDE", linestyle="--", alpha=0.7)
    ax.set_xlabel("Mask Area Ratio", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title("Dataset Distribution Comparison (KDE)", fontsize=14, pad=10)

    # Styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, which="major", alpha=0.25)
    ax.minorticks_on()
    ax.grid(True, which="minor", linestyle=":", alpha=0.1)

    # Ticks
    ax.set_xlim(0, 1)
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.tick_params(axis='both', which='major', labelsize=10)

    ax.legend(frameon=False, fontsize=10, loc="upper right")
    fig.tight_layout()

    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [5]:
plot_train_vs_test_kde(train_data, test_data, savefig_name="distbn_shift")

### Plotting #sample vs area ratios

In [6]:
def plot_area_vs_sample(
    area_ratios,
    data_type,
    savefig_name=None,
    x_tick_size=20,
):
    area_ratios = np.asarray(area_ratios, dtype=float)
    n = len(area_ratios)
    x = np.arange(1, n + 1)

    fig, ax = plt.subplots(figsize=(11, 6), dpi=300)
    ax.plot(x, area_ratios, linewidth=2, solid_capstyle="round")
    ax.set_title(f"{data_type} Dataset — Area Ratio vs Sample", pad=12)
    ax.set_xlabel("# Sample")
    ax.set_ylabel("Area Ratio")

    #styling
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.minorticks_on()
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.10, linestyle=":")

    #ticks
    step = max(1, n // x_tick_size)
    xticks = np.arange(0, n + 1, step)
    if xticks[-1] != n:
        xticks = np.append(xticks, n)
    ax.set_xticks(xticks)
    ax.set_xlim(1, n)
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0.0, 1.01, 0.1))

    # median and IQR
    q25, q50, q75 = np.percentile(area_ratios, [25, 50, 75])
    ax.axhline(q50, linestyle="--", linewidth=1.2, alpha=0.8, label=f"Median = {q50:.2f}")
    ax.fill_between([1, n], [q25, q25], [q75, q75], alpha=0.08, label="IQR (25–75%)")

    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()

    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [7]:
full_dataset = train_data+test_data
full_ratios = get_mask_area_ratio(full_dataset)
plot_area_vs_sample(full_ratios, data_type="Full", savefig_name="area_vs_sample", x_tick_size=20)

### Plotting Dice score vs #sample

In [8]:
def get_best_model_area_vs_dice(data, model_name, model_config, ckpt_path, device):
    img_transform = transforms.Compose([
            transforms.Resize(ast.literal_eval(env_vars.get("image_size")), transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor()
            ])
    msk_transform = transforms.Compose([
        transforms.Resize(ast.literal_eval(env_vars.get("mask_size")), transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
        ])
    

    model = select_model(model_name=model_name, model_config=model_config)
    
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    area_vs_score_list = []

    for i in range(len(data)):
        image, mask = img_transform(data[i][0]).to(device), msk_transform(data[i][1]).to(device)
        area_ratio = (mask == 1).sum().item()  / mask.numel()

        with torch.no_grad():
            preds = model(image.unsqueeze(0))
            if isinstance(preds, (tuple, list)):
                preds = preds[0]
            preds = preds.squeeze(0)
            dice_score = calculate_dice_score(preds=preds, targets=mask, device=device, model_name=model_name)

        area_vs_score_list.append((area_ratio, dice_score.item()))

    return area_vs_score_list

In [9]:
area_vs_score_list = get_best_model_area_vs_dice(
    data=full_dataset, 
    model_name="polyp_pvt", 
    model_config="b3", 
    ckpt_path="./pre_oversample.polyp_pvt_b3.size_384x384.pt.traindata_810.pt", 
    device="cuda")

In [10]:
areas, scores = zip(*area_vs_score_list)
plt.figure(figsize=(8, 6))
plt.scatter(areas, scores, alpha=0.6, edgecolors='k')
plt.title("Area Ratio vs Dice Score")
plt.xlabel("Area Ratio (object size relative to image)")
plt.ylabel("Dice Score")
plt.grid(True, linestyle='--', alpha=0.5)
outdir = f'.{env_vars["experiments_folder_path"]}/figs'
os.makedirs(outdir, exist_ok=True)
plt.savefig(f"{outdir}/area_vs_dice_scatter.pdf", format="pdf", bbox_inches="tight")
plt.close()


In [11]:
def plot_best_model_butterfly_mask_vs_score(n_bins, threshold, results, savefig_name):
    mask_bins = np.linspace(0.0, 1.0, n_bins + 1)
    data_bins = [[] for _ in range(n_bins)]

    for area, score in results:
        for j in range(n_bins):
            if mask_bins[j] <= area < mask_bins[j + 1]:
                data_bins[j].append(score)
                break

    butterfly_data = [
        (
            sum(score < threshold for score in bin_scores),  # below
            sum(score > threshold for score in bin_scores)   # above
        )
        for bin_scores in data_bins
    ]

    below_counts = [x[0] for x in butterfly_data]
    above_counts = [x[1] for x in butterfly_data]

    bar_width = 0.35
    index = np.arange(n_bins)

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.bar(index, -np.array(below_counts), bar_width, color='blue', label=f'Below Threshold (< DSC={threshold})')
    ax.bar(index, above_counts, bar_width, color='red', label=f'Above Threshold (> DSC={threshold})')

    ax.axhline(y=0, color='black', linestyle='--')

    for i in range(n_bins):
        ax.text(i, -below_counts[i] - 0.5, str(below_counts[i]), ha='center', va='top', fontsize=9)
        ax.text(i, above_counts[i] + 0.5, str(above_counts[i]), ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Mask Area Ratio Bins')
    ax.set_ylabel('Dice Score Counts')
    ax.set_xticks(index)
    ax.set_xticklabels([f'{mask_bins[i]:.2f}-{mask_bins[i+1]:.2f}' for i in range(n_bins)])
    plt.xticks(rotation=45)
    ax.legend()
    # ax.set_title(f"Dice Score Distribution Across Mask Size Bins (Threshold = {threshold})")

    plt.tight_layout()
    if savefig_name:
        outdir = f'.{env_vars["experiments_folder_path"]}/figs'
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [19]:
for threshold in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97]:
    plot_best_model_butterfly_mask_vs_score(n_bins=20, 
                                            threshold=threshold, 
                                            results=area_vs_score_list, 
                                            savefig_name=f"pre_oversampling_butterfly_{threshold}")

In [18]:
# def plot_best_model_butterfly_mask_vs_score_multi(
#     n_bins,
#     thresholds,
#     results,
#     savefig_name=None,
#     split_score=None,
#     figsize=(12, 8),
#     annotate=True,
# ):
#     mask_bins = np.linspace(0.0, 1.0, n_bins + 1)
#     data_bins = [[] for _ in range(n_bins)]
#     for area, score in results:
#         # place score into the appropriate area bin
#         for j in range(n_bins):
#             if mask_bins[j] <= area < mask_bins[j + 1] or (j == n_bins - 1 and np.isclose(area, 1.0)):
#                 data_bins[j].append(score)
#                 break

#     # --- Prep thresholds & score bands (disjoint) ---
#     if thresholds is None:
#         thresholds = []
#     thr = sorted(set(float(t) for t in thresholds if 0.0 < t < 1.0))
#     # band edges: [0, t1, t2, ..., tK, 1]
#     band_edges = [0.0] + thr + [1.0]
#     n_bands = len(band_edges) - 1

#     # default split score = median threshold (or 0.5 if none)
#     if split_score is None:
#         split_score = (thr[len(thr)//2] if thr else 0.5)

#     # Decide which bands go left vs right
#     # Left: bands fully below split_score (upper edge <= split_score)
#     left_band_idxs = [i for i in range(n_bands) if band_edges[i+1] <= split_score]
#     right_band_idxs = [i for i in range(n_bands) if i not in left_band_idxs]

#     # Labels for legend
#     band_labels = []
#     for i in range(n_bands):
#         lo, hi = band_edges[i], band_edges[i+1]
#         if i == 0:
#             band_labels.append(f"score < {hi:.2f}")
#         elif i == n_bands - 1:
#             band_labels.append(f"score ≥ {lo:.2f}")
#         else:
#             band_labels.append(f"{lo:.2f} ≤ score < {hi:.2f}")

#     # Count scores per (area bin, score band)
#     # counts[j][i] = count in area bin j and score band i
#     counts = np.zeros((n_bins, n_bands), dtype=int)
#     for j, bin_scores in enumerate(data_bins):
#         if not bin_scores:
#             continue
#         bs = np.asarray(bin_scores)
#         # For each band, count scores in [lo, hi) except include hi==1.0 in last band
#         for i in range(n_bands):
#             lo, hi = band_edges[i], band_edges[i+1]
#             if i == n_bands - 1:
#                 in_band = (bs >= lo) & (bs <= hi + 1e-12)  # include 1.0 safely
#             else:
#                 in_band = (bs >= lo) & (bs < hi)
#             counts[j, i] = int(np.sum(in_band))

#     # --- Plot ---
#     bar_width = 0.6
#     index = np.arange(n_bins)

#     fig, ax = plt.subplots(figsize=figsize)

#     # Stacked left (negative) for left bands
#     left_baseline = np.zeros(n_bins, dtype=int)
#     for i in left_band_idxs:
#         heights = -counts[:, i]  # negative for left side
#         ax.bar(index, heights, bar_width, bottom=left_baseline, label=band_labels[i])
#         if annotate:
#             for x, h, b in zip(index, heights, left_baseline):
#                 if h != 0:
#                     y = b + h/2
#                     ax.text(x, y, f"{abs(int(h))}", ha='center', va='center', fontsize=8)
#         left_baseline += heights  # heights are negative numbers

#     # Stacked right (positive) for right bands
#     right_baseline = np.zeros(n_bins, dtype=int)
#     for i in right_band_idxs:
#         heights = counts[:, i]
#         ax.bar(index, heights, bar_width, bottom=right_baseline, label=band_labels[i])
#         if annotate:
#             for x, h, b in zip(index, heights, right_baseline):
#                 if h != 0:
#                     y = b + h/2
#                     ax.text(x, y, f"{int(h)}", ha='center', va='center', fontsize=8)
#         right_baseline += heights

#     # Styling
#     ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
#     ax.set_xlabel('Mask Area Ratio Bins')
#     ax.set_ylabel('Dice Score Counts (stacked bands)')
#     ax.set_xticks(index)
#     ax.set_xticklabels([f'{mask_bins[i]:.2f}-{mask_bins[i+1]:.2f}' for i in range(n_bins)], rotation=45)
#     ax.legend(title="Score bands", ncol=2, fontsize=9, title_fontsize=10, frameon=False)

#     # Helpful subtitle
#     ax.set_title(
#         f"Dice Score Distribution Across Mask Size Bins\n"
#         f"Bands from thresholds={thr} | Left: < {split_score:.2f} | Right: ≥ {split_score:.2f}",
#         fontsize=12
#     )

#     plt.tight_layout()

#     if savefig_name:
#         outdir = f'.{env_vars["experiments_folder_path"]}/figs'
#         os.makedirs(outdir, exist_ok=True)
#         fig.savefig(f"{outdir}/{savefig_name}.pdf", format="pdf", bbox_inches="tight")
#         plt.close(fig)
#     else:
#         plt.show()


In [ ]:
# plot_best_model_butterfly_mask_vs_score_multi(
#     n_bins=20,
#     thresholds=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97],
#     results=area_vs_score_list,
#     savefig_name="multi_threshold_butterfly",
#     split_score=None,
#     figsize=(20, 40),
#     annotate=True)